In [6]:
# import necessary packages
import folium
from pathlib import Path
import pandas as pd
import geopandas as gpd

In [7]:
# establish paths to datasets
project_fldr = Path.cwd().parents[1]
raw_fldr = project_fldr / 'data' / 'raw'
processed_fldr = project_fldr / 'data' / 'processed'
figure_fldr = project_fldr/"figures"

greyed out because we have pulled and subset the data to be used

In [ ]:
# # access fire data via function
# def read_fires(year):
#     # SET PATH TO DATA FOLDER ON BOX DRIVE
#     if platform.system() == 'Windows':
#         box_path = Path.home() / 'Box'
#     # if mac:
#     else:
#         box_path = Path.home() / "Library" / "CloudStorage" / "Box-Box"

#     # Confirm that the box path is valid
#     if box_path.exists():
#         print(f'Box folder found at {box_path}')
#     else:
#         print(f'Failed to locate the Box folder.\nIs the BoxDrive app running?\nAre you logged in?')

#     # create path object that goes to the specific folder on Box drive
#     fire_path = box_path / 'ENVIRON 712L-Students' / 'Data' / 'BunkerAndersonTseng' / 'fire25_1.gdb'
#     print(fire_path.exists())

#     fire_shp = gpd.read_file(fire_path, layer='firep25_1')

#     # FILTER BY DESIRED YEAR
#     fire_yr = fire_shp[fire_shp['YEAR_'] == year] # this establishes the naming convention?

#     # drop rows with null values in geometry column
#     fire_yr = fire_yr.dropna(subset=['geometry'])

#     # add column for month of alarm date
#     # add a column for month of alarm date
#     # add a column for month of alarm date
#     fire_yr['Alarm Month'] = fire_yr['ALARM_DATE'].dt.month

#     fire_yr['Alarm Month Abb'] = pd.Categorical(
#         values=fire_yr['ALARM_DATE'].dt.month_name().str[:3],
#         categories=calendar.month_abbr[1:]
#     )

#     return fire_yr

In [ ]:
# # read in fires for 2020-2025
# years=[2020, 2021, 2022, 2023, 2024, 2025]

# for year in years:
#     globals()[f"fire_{year}"] = read_fires(year) # dynamically creates the five dataframes

# # fire_2020 through fire_2025 now exist

# # write a function to save them as geojson

# fires = [fire_2020, fire_2021, fire_2022, fire_2023, fire_2024, fire_2025]

# # dont forget to convert to a geographic coordinate system for explore!
# for fire, year in zip(fires, years):
#     fire = fire.to_crs("EPSG:4326")
#     fire.to_file(
#         processed_fldr / f"fire_{year}.geojson",
#         driver="GeoJSON"
#     )

Pull the data into files from geojsons

In [8]:
# california outline
ca_outline = gpd.read_file(processed_fldr/"fire_explore/ca_outline.geojson")

# pull in the fires into dfs from geojsons
fires = ["fire_2020", "fire_2021", "fire_2022", "fire_2023", "fire_2024", "fire_2025"]
years=[2020, 2021, 2022, 2023, 2024, 2025]

for fire, year in zip(fires, years):
    globals()[fire] = gpd.read_file(
        processed_fldr/ "fire_explore" / f"fire_{year}.geojson"
    )

fire_dfs = [fire_2020, fire_2021, fire_2022, fire_2023, fire_2024, fire_2025]

years = [2020, 2021, 2022, 2023, 2024, 2025]

# remove timestamp columns because they are not serializable for JSON
for fire_df in fire_dfs:
    fire_df.drop(columns=["ALARM_DATE", "CONT_DATE"], inplace=True)

/var/folders/42/6554qxxn1c9_99q7_fn53dyw0000gn/T/ipykernel_16961/1308285216.py:19: Pandas4Warning: The inplace keyword in GeoDataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  fire_df.drop(columns=["ALARM_DATE", "CONT_DATE"], inplace=True)
/var/folders/42/6554qxxn1c9_99q7_fn53dyw0000gn/T/ipykernel_16961/1308285216.py:19: Pandas4Warning: The inplace keyword in GeoDataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  fire_df.drop(columns=["ALARM_DATE", "CONT_DATE"], inplace=True)
/var/folders/42/6554qxxn1c9_99q7_fn53dyw0000gn/T/ipykernel_16961/1308285216.py:19: Pandas4Warning: The inplace keyword in GeoDataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#

In [ ]:
# DEFINE MAP PROPERTIES

minx, miny, maxx, maxy = ca_outline.total_bounds

fire_colors = {
    "2020": "#D7191C",  # Deep red
    "2021": "#E7471A",  # Red-orange
    "2022": "#F46D1F",  # Orange
    "2023": "#F99524",  # Golden orange
    "2024": "#FDC43A",  # Gold
    "2025": "#FFFF33"   # Yellow
}

generate the actual widget

In [ ]:
# make the actual explore

# generate the map
m = folium.Map(
    zoom_start=6,
    tiles=None
)

# ESRI satellite basemap
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri, Maxar, Earthstar Geographics, and the GIS User Community",
    name="Satellite Imagery",
    overlay=False,
    control=False
).add_to(m)

# this should leave us so that the initial zoom is the outline of california
m.fit_bounds([
    [miny, minx],  # Southwest corner
    [maxy, maxx]   # Northeast corner
])

# add CA outline
folium.GeoJson(
    ca_outline.to_json(),
    style_function=lambda x: {
        "color": "black",
        "fillOpacity": 0,
        "weight": 2
    },
    control= False # always on
).add_to(m)

# for loop to add in the fires
for fire_df, year in zip(fire_dfs, years):
    folium.GeoJson(
        fire_df.to_json(),
        name=f"{year} Fires",
        style_function=lambda x, year=year: {
            "color": fire_colors[str(year)],
            "fillColor": fire_colors[str(year)],
            "fillOpacity": 0.9,
            "weight": 1.5
        },
        control=True
    ).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

m